<a href="https://colab.research.google.com/github/Kingelanci/graphysics/blob/main/Peptline_v9_SingleFilterMode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ᾞ NutrAI Pipeline v9 — Single-Filter Mode (Table S2)
## STOP after STEP 3 — produces the 6 standalone counts for the SPAFE paper

**Difference vs standard v9 pipeline:**
- Standard pipeline: continues with ESM-2 mining, Boltz-2, gold candidates.
- This version: **stops after STEP 3 (hydrolysis)** and calculates the 6 standalone counts for Table S2.


**For the SPAFE/MDM2 paper (Zingiber officinale → SPAFESTWDILK):** at STEP 0b select DATABASE=`B` (phytotherapeutics). The ESM-2 mining target does not affect the STEP 3b counts (filters are target-independent).

**Run in order:**
1. STEP 0 — dependencies
2. STEP 0b — select DB/target
3. STEP 0c — load DB
4. STEP 1a / 1b — configure targets / proteomes
5. STEP 2 — load proteomes
6. STEP 3 — enzymatic hydrolysis → generates `peptides_hydrolysis.csv`
7. **STEP 3b — Single-Filter Mode → final output for Minoo**

No ESM-2, no Boltz-2, no SQLite database.

In [17]:
# ============================================================
# STEP 0 — INSTALLAZIONE DIPENDENZE
# ============================================================
import subprocess, sys

packages = [
    "transformers",
    "torch",
    "biopython",
    "pandas",
    "matplotlib",
    "seaborn",
    "tqdm",
    "pyyaml",
]

print("📦 Installing dependencies...")
for pkg in packages:
    try:
        __import__(pkg.split("[")[0] if pkg != "pyyaml" else "yaml")
        print(f"   ✅ {pkg} already installed")
    except ImportError:
        print(f"   📥 Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"   ✅ {pkg} installed")

import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n🖥️ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("\n⚠️ No GPU! Runtime → Change runtime type → GPU")

print("\n✅ Setup completed!")


📦 Installing dependencies...
   ✅ transformers already installed
   ✅ torch already installed
   📥 Installing biopython...
   ✅ biopython installed
   ✅ pandas already installed
   ✅ matplotlib already installed
   ✅ seaborn already installed
   ✅ tqdm already installed
   ✅ pyyaml already installed

🖥️ GPU: Tesla T4 (15.6 GB)

✅ Setup completed!


In [18]:
# ╔══════════════════════════════════════════════════════════╗
# ║           SELECT DATABASE AND TARGET                     ║
# ╚══════════════════════════════════════════════════════════╝

print("""
┌─────────── DATABASE ───────────┐
│  A = Food                      │
│  B = Phytotherapeutics         │
│  C = Food Waste                │
│  G = ALL                       │
│                                │
│  Multiple: A,B  or A,C         │
└────────────────────────────────┘
""")
DATABASE = input("DATABASE (A/B/C/G or combination e.g., A,B): ").strip().upper() or "B"

print(f"""
┌─────────── TARGET ─────────────────────────┐
│  H = IL-11 (longevity)                     │
│  J = ACE (cardiovascular)                  │
│  K = EphA2 (GBM)                           │
│  L = IL-13Rα2 (GBM)                        │
│  M = MDM2 (p53)                            │
│  O = ALL 5                                 │
│                                            │
│  Multiple: H,M  or H,J                     │
└────────────────────────────────────────────┘
""")
TARGET = input("TARGET (H/J/K/L/M/O or combination e.g., H,K): ").strip().upper() or "K"

# ── Parsing selezione ──
DB_LETTER_MAP = {"A": ["alimenti"], "B": ["fitoterapici"], "C": ["scarti"], "G": ["alimenti", "fitoterapici", "scarti"]}
TARGET_LETTER_MAP = {"H": ["IL11_IL11Ra"], "J": ["ACE"], "K": ["EphA2"], "L": ["IL13Ra2"], "M": ["MDM2"], "O": ["IL11_IL11Ra", "ACE", "EphA2", "IL13Ra2", "MDM2"]}

# Database: supporta singolo, multiplo (A,B) o G
SELECTED_DBS = []
for letter in DATABASE.replace(" ", "").split(","):
    SELECTED_DBS.extend(DB_LETTER_MAP.get(letter, []))
SELECTED_DBS = list(dict.fromkeys(SELECTED_DBS))  # deduplica

# Target: supporta singolo, multiplo (H,K) o O
SELECTED_TARGETS = []
for letter in TARGET.replace(" ", "").split(","):
    SELECTED_TARGETS.extend(TARGET_LETTER_MAP.get(letter, []))
SELECTED_TARGETS = list(dict.fromkeys(SELECTED_TARGETS))  # deduplica

# Riepilogo
DB_NAMES = {"alimenti": "🥦 Food", "fitoterapici": "🌿 Phytotherapeutics", "scarti": "🗑️ Waste"}
TARGET_NAMES = {"IL11_IL11Ra": "IL-11", "ACE": "ACE", "EphA2": "EphA2", "IL13Ra2": "IL-13Rα2", "MDM2": "MDM2"}

print(f"""
{'='*50}
  DATABASE: {' + '.join(DB_NAMES.get(d, d) for d in SELECTED_DBS)}
  TARGET:   {', '.join(TARGET_NAMES.get(t, t) for t in SELECTED_TARGETS)}
{'='*50}
""")



┌─────────── DATABASE ───────────┐
│  A = Food                      │
│  B = Phytotherapeutics         │
│  C = Food Waste                │
│  G = ALL                       │
│                                │
│  Multiple: A,B  or A,C         │
└────────────────────────────────┘

DATABASE (A/B/C/G or combination e.g., A,B): B

┌─────────── TARGET ─────────────────────────┐
│  H = IL-11 (longevity)                     │
│  J = ACE (cardiovascular)                  │
│  K = EphA2 (GBM)                           │
│  L = IL-13Rα2 (GBM)                        │
│  M = MDM2 (p53)                            │
│  O = ALL 5                                 │
│                                            │
│  Multiple: H,M  or H,J                     │
└────────────────────────────────────────────┘

TARGET (H/J/K/L/M/O or combination e.g., H,K): M

  DATABASE: 🌿 Phytotherapeutics
  TARGET:   MDM2



In [19]:
# ============================================================
# STEP 0c — SETUP AMBIENTE E CARICAMENTO DATABASE
# ============================================================
import os, glob

# Ambiente
if os.path.exists('/workspace'):
    ENV = 'runpod'
    BASE_DIR = '/workspace'
    SAVE_DIR = '/workspace/nutrai_results'
elif os.path.exists('/content'):
    ENV = 'colab'
    BASE_DIR = '/content'
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        SAVE_DIR = '/content/drive/MyDrive/NutrAI_results'
    except:
        SAVE_DIR = None
else:
    ENV = 'local'
    BASE_DIR = os.path.expanduser('~')
    SAVE_DIR = os.path.join(BASE_DIR, 'nutrai_results')

print(f"🚀 {ENV}")

# Directory
WORK_DIR = os.path.join(BASE_DIR, "nutrai_pipeline")
DATA_DIR = os.path.join(WORK_DIR, "data")
RESULTS_DIR = os.path.join(WORK_DIR, "results")
BOLTZ_DIR = os.path.join(WORK_DIR, "boltz2_export")
DB_PATH = os.path.join(WORK_DIR, "nutrai.db")
for d in [WORK_DIR, DATA_DIR, RESULTS_DIR, BOLTZ_DIR]:
    os.makedirs(d, exist_ok=True)
if SAVE_DIR:
    os.makedirs(SAVE_DIR, exist_ok=True)

# ── Mappa database ──
DB_MAP = {
    "A": ["alimenti"], "B": ["fitoterapici"], "C": ["scarti"],
    "D": ["alimenti", "fitoterapici"], "E": ["alimenti", "scarti"],
    "F": ["fitoterapici", "scarti"], "G": ["alimenti", "fitoterapici", "scarti"],
}
DB_CATEGORIES_MAP = {
    "alimenti": ["alimenti_cereali", "alimenti_legumi", "alimenti_semi_oleosi", "alimenti_pseudocereali", "alimenti_microalghe"],
    "fitoterapici": ["fito_funghi", "fito_piante"],
    "scarti": ["scarti_bucce_frutta", "scarti_bucce_ortaggi", "scarti_gusci_semi"],
}
DB_NAMES = {"alimenti": "🥦 Food", "fitoterapici": "🌿 Phytotherapeutics", "scarti": "🗑️ Waste"}

# ── Cerca e estrai tar.gz ──
print("\n🔍 Searching for tar.gz databases...\n")
tar_files = set()
for search in [BASE_DIR, SAVE_DIR or "", WORK_DIR]:
    if search and os.path.exists(search):
        for tf in glob.glob(os.path.join(search, "db_*.tar.gz")):
            tar_files.add(tf)
        for tf in glob.glob(os.path.join(search, "**", "db_*.tar.gz"), recursive=True):
            tar_files.add(tf)

FOUND_DBS = []
for tf in sorted(tar_files):
    fname = os.path.basename(tf)
    os.system(f'tar xf "{tf}" -C {DATA_DIR} 2>/dev/null')
    os.system(f'tar xf "{tf}" -C / 2>/dev/null')
    for db_key in ["alimenti", "fitoterapici", "scarti", "TUTTI"]:
        if db_key in fname:
            FOUND_DBS.append(db_key)
            print(f"   ✅ {DB_NAMES.get(db_key, fname)}: {fname}")
            break

# ── Determina categorie attive ──
if DATABASE.upper() == "AUTO":
    # Auto-detect da tar.gz trovati
    if "TUTTI" in FOUND_DBS:
        SELECTED_DBS = ["alimenti", "fitoterapici", "scarti"]
    else:
        SELECTED_DBS = [d for d in FOUND_DBS if d != "TUTTI"]
else:
    SELECTED_DBS = DB_MAP.get(DATABASE.upper(), ["alimenti", "fitoterapici", "scarti"])

SELECTED_CATEGORIES = []
for db in SELECTED_DBS:
    SELECTED_CATEGORIES.extend(DB_CATEGORIES_MAP.get(db, []))

if not SELECTED_CATEGORIES:
    fasta_count = len(glob.glob(os.path.join(DATA_DIR, "*.fasta")))
    if fasta_count > 0:
        SELECTED_CATEGORIES = [cat for cats in DB_CATEGORIES_MAP.values() for cat in cats]
        print(f"   ℹ️ {fasta_count} fasta found — using all")
    else:
        raise FileNotFoundError("No database found! Upload tar.gz files to /workspace/")

# ── Mappa target ──
TARGET_MAP = {
    "H": ["IL11_IL11Ra"], "J": ["ACE"],
    "K": ["EphA2"], "L": ["IL13Ra2"],
    "M": ["MDM2"],
    "O": ["IL11_IL11Ra", "ACE", "EphA2", "IL13Ra2", "MDM2"],
}
SELECTED_TARGETS = TARGET_MAP.get(TARGET.upper(), ["IL11_IL11Ra"])

# ── Riepilogo ──
n_fasta = len(glob.glob(os.path.join(DATA_DIR, "*.fasta")))
print(f"\n{'='*50}")
print(f"  DATABASE: {' + '.join(DB_NAMES.get(d, d) for d in SELECTED_DBS)}")
print(f"  TARGET:   {', '.join(SELECTED_TARGETS)}")
print(f"  FASTA:    {n_fasta} files found")
print(f"{'='*50}")


🚀 runpod

🔍 Searching for tar.gz databases...


  DATABASE: 🌿 Phytotherapeutics
  TARGET:   MDM2
  FASTA:    22 files found


## ⚙️ STEP 1 — Target and Proteome Configuration
Defining the 5 molecular targets and 61 proteomes (food + phytotherapeutics + waste).
Each target has: reference sequence, PDB ID, receptor sequence for Boltz-2.

In [20]:
# ============================================================
# STEP 1a — CONFIGURAZIONE TARGET MOLECOLARI
# ============================================================
import os
import json


# ── TARGET MOLECOLARI ──────────────────────────────────────
# Ogni target ha:
#   - ref_peptide: il peptide di riferimento (quello che vogliamo mimare)
#   - receptor_pdb: il PDB ID del recettore per il docking
#   - receptor_seq: la sequenza del recettore (catena da usare in Boltz-2)
#   - action: "inhibit" o "activate"
#   - description: breve descrizione

TARGETS = {
    "IL11_IL11Ra": {
        "name": "IL-11 / IL-11Rα",
        "description": "Anti-inflammaging. Blocco interazione IL-11/IL-11Rα. +25% lifespan (Nature 2024)",
        "ref_peptide": None,  # Nessun peptide di riferimento noto — usiamo embedding del sito di legame
        "receptor_pdb": "6O4P",
        "action": "inhibit",
        "binding_site_residues": "FESLLEQKFCAKDY",  # Residui chiave interfaccia IL-11/IL-11Rα
        # Sequenza IL-11 umana (P20809, UniProt) — il peptide deve legarsi a QUESTA per bloccarla
        "receptor_seq": (
            "PGPPPGPPRVSPDPRAELDSTVLLTRSLLADTRQLAAQLRDKFPADGDHNLDSLPTLAMS"
            "AGALGALQLPGVLTRLRADLLSYLRHVQWLRRAGGSSLKTLEPELGTLQARLDRLLRRL"
            "QLLMSRLALPQPPPDPPAPPLAPPSSAWGGIRAAHAILGGLHLTLDWAVRGLLLLKTRL"
        ),
        "priority": 1,
    },

    "ACE": {
        "name": "ACE (Angiotensin Converting Enzyme)",
        "description": "Cardiovascolare. Benchmark: peptidi anti-ACE già validati (IPP, VPP).",
        "ref_peptide": "IPP",  # Isoleucina-Prolina-Prolina — peptide anti-ACE noto
        "receptor_pdb": "1O86",
        "action": "inhibit",
        "binding_site_residues": "HHEAEYQHKHYE",  # His383,His387,Glu411,Ala354,Glu384,Tyr523,Gln281,His353,Lys511,His513,Tyr520,Glu162
        # Sequenza ACE umano dominio C (da PDB 1O86, catena A)
        "receptor_seq": (
            "DEAGAQLFAQSYNSSAEQVLFQSVAASWAHDTNITAENARRQEEAALLSQEFAEAWGQK"
            "AKELYEPIWQNFTDPQLRRIIGAVRTLGSANLPLAKRQQYNALLSNMSRIYSTAKVCLP"
            "NKTATCWSLDPDLTNILASSRSYAMLLFAWEGWHNAAGIPLKPLYEDFTALSNEAYKQD"
            "GFTDTGAYWRSWYNSPTFEDDLEHLYQQLEPLYLNVDDLELDMSLNLIPDNPYGEPWN"
            "EDDKHTIMQKAMEAYFKGDYGKQKLRFQKDHEKFSNQYESHFHGKLNFANPNKTKDN"
            "QEFLNLNPKDEEIMKVDNVSQVNNYFAVKNNLQPTLQNAQPDQKVIYNPVNDLQPGM"
            "FSFNPQGKPGELQYLLATQPELDSKLASKTRYNYLRATLQNYTESQPIQNSLPQGFIS"
            "GQAQPGDKFLEAIRTGSTFKLVLNQNKDSREALAQALKQSMKDAAFEKEIITHKLDQW"
            "LSDQHVDTFAGNKFKQLEGLFEQFQLPAQNQWQHGKQEPTQRRSLELGAYPGAPELL"
            "LGGSSFELLNFKQPEKQRSFQETLIDQNKKFPQVTRGQVKNFPNLFENENIGYRYYNF"
            "NIFEGQCYYFHKALFKAWIEEVSKQFNQNTQDYGLYQGAYQEEERLKKLCELYAKVLG"
            "SLKKNPKNFMKLQPLLESQATIWFLLNKDLLLNQFTPNSPRQSPFVSSMKDLATFSKN"
            "QFNLTTQVNLFEQSKQVSHRYIFVDEKRQLYLQKVSQPYNQNQTGERDLSSHLAMYC"
        ),
        "priority": 3,
    },
        "EphA2": {
        "name": "EphA2 (Ephrin type-A receptor 2)",
        "description": "Oncologia GBM. Driver primario, sovraespresso 90% glioblastomi. Approccio agonistico.",
        "ref_peptide": "KLSEKFQRFTPFTLG",  # G-H loop Ephrin-A1 (P20827) — ligando naturale
        "receptor_pdb": "3HEI",
        "action": "activate",
        "binding_site_residues": "KLSEKFQRFTPFTLG",  # G-H loop 15 aa, Phe108/111/114 ancore
        "receptor_seq": (
            "EVLWMCNPKKNRNEPIVIRGATVTFKCEASNKGPNSFHQGEHFCQADITSNLTIQQSV"
            "FSKDYPDYFEDPRQPFYIREDRLMQHFVHYTGAAGKTWYIDGKELPQHVKGIAAGNTY"
            "YNEKFMINKLPTLYKNPVSEDQFPNSSRFQIEINTTPPMVEGQYEIKVSFLDKEIIRPK"
            "EVNWYLNHTFVPELQLDECNIMHYKADGGKAMCKFCEQPGQDSGELNICGKCEQPMYSA"
            "VVTKHAQCQRVDNECGQFNECLGQVNVSSSEGPICGWDGFDCAADQFSSERNCRTCGA"
            "GFHCEKNINESFPDPAGKCSAPNQGPAEDSRIVSYQIGDTQMFNKDLKDPNEYSGQLD"
            "KDIYQKTYIQSVALNPETLPPDFRDRLSSSRHQAFLAEFQDAERYILELAHFDAEKAME"
            "LNHSIRDKQPVFNQTIYQHLMRHEYMPNVLRFANRDVYEERGSKYPCEAYMTFGRKLC"
            "LPEQFGQLKAVFVRTPPYSDESNQAERNSSELQIS"
        ),
        "priority": 2,
    },
    "IL13Ra2": {
        "name": "IL-13Rα2 (GBM marker)",
        "description": "GBM. Selettivo per glioblastoma, quasi assente nei tessuti normali.",
        "ref_peptide": None,
        "receptor_pdb": "3LB6",
        "action": "inhibit",
        "binding_site_residues": "CYYKNREDLEF",
        "receptor_seq": (
            "CITYYKNREDLEFCPNTHSFHCIDFYIKKINPGQEAIYVMELEIQDKIFSDYQKHQEP"
            "CFLKIQFCDSGYEVQWYKNRRQPKETISTVEFLNDGDGFVTQNYRCSEGNPVKQYPE"
            "EKQIMIDIFHPSVFHGGSFCYVCQALLLNNSHQGYNSVAIRNAETLSYDEQGGSIRSK"
        ),
        "priority": 5,
    },

}

# Applica selezione target
active_targets = {k: v for k, v in TARGETS.items() if k in SELECTED_TARGETS}

# Peptidi di riferimento: usa binding site se ref_peptide è None
for tname, tdata in active_targets.items():
    if tdata["ref_peptide"] is None:
        tdata["ref_peptide"] = tdata["binding_site_residues"]

print(f"\n🎯 Target attivi: {len(active_targets)} su {len(TARGETS)}")
for tname, tdata in sorted(active_targets.items(), key=lambda x: x[1]["priority"]):
    print(f"  [{tdata['priority']}] {tdata['name']}")
    print(f"      {tdata['description']}")
    print(f"      PDB: {tdata['receptor_pdb']} | Ref: {tdata['ref_peptide']}")



🎯 Target attivi: 0 su 4


In [21]:
# ============================================================
# STEP 1b — PROTEOMI (da PATCH DEFINITIVA, 3 database)
# ============================================================

PROTEOMES = {
    "Triticum_aestivum": {
        "name": "Grano tenero",
        "organism": "Triticum aestivum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000019116%29",
        "priority": "alta",
        "category": "alimenti_cereali",
        "est_proteins": 130692,
        "proteome_id": "UP000019116",
        "note": "Proteoma ref. Glutenine, gliadine, LTP."
    },
    "Oryza_sativa": {
        "name": "Riso",
        "organism": "Oryza sativa subsp. japonica",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000059680%29",
        "priority": "alta",
        "category": "alimenti_cereali",
        "est_proteins": 48847,
        "proteome_id": "UP000059680",
        "note": "FIX: proteoma completo (era 4,197 con filtro reviewed). Orizanine, ACE-inibitori."
    },
    "Zea_mays": {
        "name": "Mais",
        "organism": "Zea mays",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000007305%29",
        "priority": "media",
        "category": "alimenti_cereali",
        "est_proteins": 63237,
        "proteome_id": "UP000007305",
        "note": "OK. Zeine, proteine di riserva."
    },
    "Hordeum_vulgare": {
        "name": "Orzo",
        "organism": "Hordeum vulgare",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000011116%29",
        "priority": "media",
        "category": "alimenti_cereali",
        "est_proteins": 35907,
        "proteome_id": "UP000011116",
        "note": "OK. Ordeine, lunasin-like."
    },
    "Avena_sativa": {
        "name": "Avena",
        "organism": "Avena sativa",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A4498%29",
        "priority": "media",
        "category": "alimenti_cereali",
        "est_proteins": 95437,
        "proteome_id": "N/A (taxonomy query)",
        "note": "FIX: proteome_id originale inesistente. Avenine, beta-glucanasi."
    },
    "Glycine_max": {
        "name": "Soia",
        "organism": "Glycine max",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000008827%29",
        "priority": "alta",
        "category": "alimenti_legumi",
        "est_proteins": 74859,
        "proteome_id": "UP000008827",
        "note": "FIX: proteoma completo (era 439 con filtro reviewed). Lunasin, BBI, glicinina."
    },
    "Cicer_arietinum": {
        "name": "Cece",
        "organism": "Cicer arietinum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A3827%29",
        "priority": "alta",
        "category": "alimenti_legumi",
        "est_proteins": 31239,
        "proteome_id": "UP000023160 (taxonomy query per completezza)",
        "note": "FIX: proteome ref ha solo 2,670. Taxonomy query dà 31,239."
    },
    "Pisum_sativum": {
        "name": "Pisello",
        "organism": "Pisum sativum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000325105%29",
        "priority": "alta",
        "category": "alimenti_legumi",
        "est_proteins": 3756,
        "proteome_id": "UP000325105",
        "note": "OK. Albumine, legumine."
    },
    "Arachis_hypogaea": {
        "name": "Arachide",
        "organism": "Arachis hypogaea",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A3818%29",
        "priority": "alta",
        "category": "alimenti_legumi",
        "est_proteins": 102020,
        "proteome_id": "UP000321265 (taxonomy query per completezza)",
        "note": "FIX: proteome ref ha solo 4,356. Taxonomy dà 102,020. Pellicina = scarto."
    },
    "Lupinus_albus": {
        "name": "Lupino",
        "organism": "Lupinus albus",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A3870%29",
        "priority": "alta",
        "category": "alimenti_legumi",
        "est_proteins": 38111,
        "proteome_id": "N/A (taxonomy query)",
        "note": "FIX: proteome_id originale inesistente. Conglutine, ipoglicemizzanti."
    },
    "Phaseolus_vulgaris": {
        "name": "Fagiolo",
        "organism": "Phaseolus vulgaris",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000000226%29",
        "priority": "media",
        "category": "alimenti_legumi",
        "est_proteins": 30854,
        "proteome_id": "UP000000226",
        "note": "OK. Fasoline, lectine."
    },
    "Vigna_unguiculata": {
        "name": "Fagiolo dall'occhio",
        "organism": "Vigna unguiculata",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A3917%29",
        "priority": "media",
        "category": "alimenti_legumi",
        "est_proteins": 39956,
        "proteome_id": "UP000234680 (taxonomy query per completezza)",
        "note": "FIX: proteome ref ha solo 1,698. Taxonomy dà 39,956."
    },
    "Helianthus_annuus": {
        "name": "Girasole",
        "organism": "Helianthus annuus",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000215914%29",
        "priority": "media",
        "category": "alimenti_semi_oleosi",
        "est_proteins": 88555,
        "proteome_id": "UP000215914",
        "note": "OK. SFTI-1 (peptide ciclico naturale!)."
    },
    "Cannabis_sativa": {
        "name": "Canapa",
        "organism": "Cannabis sativa",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A3483%29",
        "priority": "alta",
        "category": "alimenti_semi_oleosi",
        "est_proteins": 79317,
        "proteome_id": "UP000563258 (taxonomy query per completezza)",
        "note": "FIX: proteome ref ha solo 5,308. Taxonomy dà 79,317. Edestina, albumina 2S."
    },
    "Sesamum_indicum": {
        "name": "Sesamo",
        "organism": "Sesamum indicum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A4182%29",
        "priority": "media",
        "category": "alimenti_semi_oleosi",
        "est_proteins": 29855,
        "proteome_id": "UP000260353 (taxonomy query per completezza)",
        "note": "FIX: proteome ref ha solo 4,420. Taxonomy dà 29,855."
    },
    "Cucurbita_maxima": {
        "name": "Zucca",
        "organism": "Cucurbita maxima",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A3661%29",
        "priority": "bassa",
        "category": "alimenti_semi_oleosi",
        "est_proteins": 34748,
        "proteome_id": "UP000311284 (taxonomy query per completezza)",
        "note": "FIX: proteome ref ha solo 3,642. Taxonomy dà 34,748. Semi di zucca."
    },
    "Chenopodium_quinoa": {
        "name": "Quinoa",
        "organism": "Chenopodium quinoa",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A63459%29",
        "priority": "alta",
        "category": "alimenti_pseudocereali",
        "est_proteins": 34188,
        "proteome_id": "N/A (taxonomy query)",
        "note": "FIX: proteome_id originale inesistente. Chenopodin, saponine."
    },
    "Arthrospira_platensis": {
        "name": "Spirulina",
        "organism": "Arthrospira platensis",
        "url": "https://rest.uniprot.org/uniprotkb/stream?query=organism_id:118562&format=fasta",
        "priority": "alta",
        "category": "alimenti_microalghe",
        "est_proteins": 4904,
        "proteome_id": "UP001547885",
        "note": "OK. C-phycocyanin, SYSQACHNR validato."
    },
    "Chlorella_vulgaris": {
        "name": "Chlorella",
        "organism": "Chlorella vulgaris",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A3077%29",
        "priority": "alta",
        "category": "alimenti_microalghe",
        "est_proteins": 9831,
        "proteome_id": "N/A (taxonomy query — vecchio ID era del pisello!)",
        "note": "FIX CRITICO: proteome_id era UP000325105 = Pisum sativum! Corretto."
    },
    "Pleurotus_ostreatus": {
        "name": "Pleurotus",
        "organism": "Pleurotus ostreatus",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000623687%29",
        "priority": "alta",
        "category": "fito_funghi",
        "est_proteins": 11624,
        "proteome_id": "UP000623687",
        "note": "OK. Pleurotina, lovastatina."
    },
    "Cordyceps_militaris": {
        "name": "Cordyceps",
        "organism": "Cordyceps militaris",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000001610%29",
        "priority": "alta",
        "category": "fito_funghi",
        "est_proteins": 9651,
        "proteome_id": "UP000001610",
        "note": "OK. Cordycepina, performance, longevità."
    },
    "Ophiocordyceps_sinensis": {
        "name": "Cordyceps sinensis",
        "organism": "Ophiocordyceps sinensis",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000557566%29",
        "priority": "alta",
        "category": "fito_funghi",
        "est_proteins": 9923,
        "proteome_id": "UP000557566",
        "note": "OK. Caterpillar fungus tradizionale."
    },
    "Grifola_frondosa": {
        "name": "Maitake",
        "organism": "Grifola frondosa",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000092993%29",
        "priority": "alta",
        "category": "fito_funghi",
        "est_proteins": 14985,
        "proteome_id": "UP000092993",
        "note": "OK. Grifolan (beta-glucano), immunomodulazione."
    },
    "Lentinula_edodes": {
        "name": "Shiitake",
        "organism": "Lentinula edodes",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000188533%29",
        "priority": "alta",
        "category": "fito_funghi",
        "est_proteins": 12046,
        "proteome_id": "UP000188533",
        "note": "OK. Lentinano (beta-glucano)."
    },
    "Trametes_versicolor": {
        "name": "Coda di tacchino",
        "organism": "Trametes versicolor",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000054317%29",
        "priority": "media",
        "category": "fito_funghi",
        "est_proteins": 1022,
        "proteome_id": "UP000054317",
        "note": "OK. PSK/PSP polisaccaridi."
    },
    "Ganoderma_lucidum": {
        "name": "Reishi",
        "organism": "Ganoderma lucidum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A5315%29",
        "priority": "media",
        "category": "fito_funghi",
        "est_proteins": 360,
        "proteome_id": "N/A",
        "note": "Proteoma limitato. Triterpeni noti."
    },
    "Zingiber_officinale": {
        "name": "Zenzero",
        "organism": "Zingiber officinale",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A94328%29",
        "priority": "alta",
        "category": "fito_piante",
        "est_proteins": 71836,
        "proteome_id": "N/A",
        "note": "OK. Proteoma enorme. Gingeroli noti, peptidi inesplorati."
    },
    "Panax_ginseng": {
        "name": "Ginseng",
        "organism": "Panax ginseng",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A4054%29",
        "priority": "alta",
        "category": "fito_piante",
        "est_proteins": 840,
        "proteome_id": "N/A",
        "note": "Proteoma limitato. Ginsenosidi noti. Adattogeno."
    },
    "Camellia_sinensis": {
        "name": "Tè verde",
        "organism": "Camellia sinensis",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000306102%29",
        "priority": "alta",
        "category": "fito_piante",
        "est_proteins": 30052,
        "proteome_id": "UP000306102",
        "note": "NUOVO. EGCG noto, peptidi inesplorati. TePigal."
    },
    "Artemisia_annua": {
        "name": "Artemisia",
        "organism": "Artemisia annua",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000245207%29",
        "priority": "alta",
        "category": "fito_piante",
        "est_proteins": 66067,
        "proteome_id": "UP000245207",
        "note": "NUOVO. Artemisinina, anti-tumorale attivo."
    },
    "Olea_europaea": {
        "name": "Olivo",
        "organism": "Olea europaea",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000594638%29",
        "priority": "alta",
        "category": "fito_piante",
        "est_proteins": 78352,
        "proteome_id": "UP000594638",
        "note": "NUOVO. Oleuropeina, idrossitirosolo. Anti-inflammaging."
    },
    "Curcuma_longa": {
        "name": "Curcuma",
        "organism": "Curcuma longa",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A136217%29",
        "priority": "alta",
        "category": "fito_piante",
        "est_proteins": 244,
        "proteome_id": "N/A",
        "note": "Proteoma limitato (244). Curcuminoidi noti."
    },
    "Glycyrrhiza_glabra": {
        "name": "Liquirizia",
        "organism": "Glycyrrhiza glabra",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A49827%29",
        "priority": "bassa",
        "category": "fito_piante",
        "est_proteins": 346,
        "proteome_id": "N/A",
        "note": "Proteoma limitato. Glicirrizina nota."
    },
    "Moringa_oleifera": {
        "name": "Moringa",
        "organism": "Moringa oleifera",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A3735%29",
        "priority": "media",
        "category": "fito_piante",
        "est_proteins": 192,
        "proteome_id": "N/A",
        "note": "Proteoma limitato. Peptidi antimicrobici noti."
    },
    "Silybum_marianum": {
        "name": "Cardo mariano",
        "organism": "Silybum marianum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A92921%29",
        "priority": "bassa",
        "category": "fito_piante",
        "est_proteins": 183,
        "proteome_id": "N/A",
        "note": "Proteoma limitato. Silimarina nota. Epatoprotettore."
    },
    "Centella_asiatica": {
        "name": "Centella",
        "organism": "Centella asiatica",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A48106%29",
        "priority": "bassa",
        "category": "fito_piante",
        "est_proteins": 170,
        "proteome_id": "N/A",
        "note": "Proteoma limitato. Neuroprotettore, cicatrizzante."
    },
    "Reynoutria_japonica": {
        "name": "Poligono del Giappone",
        "organism": "Reynoutria japonica",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A488216%29",
        "priority": "bassa",
        "category": "fito_piante",
        "est_proteins": 174,
        "proteome_id": "N/A",
        "note": "Proteoma limitato. Fonte principale di resveratrolo."
    },
    "Astragalus_membranaceus": {
        "name": "Astragalo",
        "organism": "Astragalus membranaceus",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28taxonomy_id%3A1862641%29",
        "priority": "bassa",
        "category": "fito_piante",
        "est_proteins": 76,
        "proteome_id": "N/A",
        "note": "Proteoma molto limitato (76). Immunomodulazione."
    },
    "Cinnamomum_verum": {
        "name": "Cannella",
        "organism": "Cinnamomum verum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?query=taxonomy_id:13424&format=fasta",
        "priority": "media",
        "category": "fito_piante",
        "est_proteins": 172,
        "proteome_id": "N/A",
        "note": "Limitato. Cinnamaldeide nota."
    },
    "Crocus_sativus": {
        "name": "Zafferano",
        "organism": "Crocus sativus",
        "url": "https://rest.uniprot.org/uniprotkb/stream?query=taxonomy_id:82528&format=fasta",
        "priority": "media",
        "category": "fito_piante",
        "est_proteins": 315,
        "proteome_id": "N/A",
        "note": "Limitato. Crocina/safranale noti."
    },
    "Vanilla_planifolia": {
        "name": "Vaniglia",
        "organism": "Vanilla planifolia",
        "url": "https://rest.uniprot.org/uniprotkb/stream?query=proteome:UP000636800&format=fasta",
        "priority": "media",
        "category": "fito_piante",
        "est_proteins": 51925,
        "proteome_id": "UP000636800",
        "note": "Vanillina nota. Proteoma completo."
    },
    "Solanum_lycopersicum": {
        "name": "Pomodoro (buccia)",
        "organism": "Solanum lycopersicum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000004994%29",
        "priority": "alta",
        "category": "scarti_bucce_frutta",
        "est_proteins": 34663,
        "proteome_id": "UP000004994",
        "note": "FIX: proteoma completo (era 511 con filtro reviewed). Difensine, PR proteins."
    },
    "Vitis_vinifera": {
        "name": "Uva/Vite (vinacce)",
        "organism": "Vitis vinifera",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000009183%29",
        "priority": "alta",
        "category": "scarti_bucce_frutta",
        "est_proteins": 29758,
        "proteome_id": "UP000009183",
        "note": "FIX: proteoma completo (era 187 con filtro reviewed). Taumatine-like, LTP."
    },
    "Solanum_tuberosum": {
        "name": "Patata (buccia)",
        "organism": "Solanum tuberosum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000011115%29",
        "priority": "media",
        "category": "scarti_bucce_ortaggi",
        "est_proteins": 53106,
        "proteome_id": "UP000011115",
        "note": "OK. Patatine (inibitori proteasici). Buccia ricca di difensine."
    },
    "Punica_granatum": {
        "name": "Melograno (buccia)",
        "organism": "Punica granatum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000233551%29",
        "priority": "alta",
        "category": "scarti_bucce_frutta",
        "est_proteins": 50426,
        "proteome_id": "UP000233551",
        "note": "NUOVO. Bucce ricchissime di proteine, antiossidanti."
    },
    "Citrus_sinensis": {
        "name": "Arancia (buccia)",
        "organism": "Citrus sinensis",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000829398%29",
        "priority": "alta",
        "category": "scarti_bucce_frutta",
        "est_proteins": 80120,
        "proteome_id": "UP000829398",
        "note": "NUOVO. Bucce agrumi, flavonoidi noti."
    },
    "Malus_domestica": {
        "name": "Mela (buccia/torsolo)",
        "organism": "Malus domestica",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000290289%29",
        "priority": "media",
        "category": "scarti_bucce_frutta",
        "est_proteins": 42478,
        "proteome_id": "UP000290289",
        "note": "NUOVO. Bucce e torsoli."
    },
    "Musa_acuminata": {
        "name": "Banana (buccia)",
        "organism": "Musa acuminata",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000012960%29",
        "priority": "media",
        "category": "scarti_bucce_frutta",
        "est_proteins": 40347,
        "proteome_id": "UP000012960",
        "note": "NUOVO. [REF] Bucce di banana."
    },
    "Persea_americana": {
        "name": "Avocado (semi/buccia)",
        "organism": "Persea americana",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP001234297%29",
        "priority": "media",
        "category": "scarti_bucce_frutta",
        "est_proteins": 36165,
        "proteome_id": "UP001234297",
        "note": "NUOVO. [REF] Semi e bucce di avocado."
    },
    "Ananas_comosus": {
        "name": "Ananas (buccia/torsolo)",
        "organism": "Ananas comosus",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000515123%29",
        "priority": "media",
        "category": "scarti_bucce_frutta",
        "est_proteins": 29712,
        "proteome_id": "UP000515123",
        "note": "NUOVO. Bromelina nel torsolo."
    },
    "Prunus_persica": {
        "name": "Pesca (nocciolo/buccia)",
        "organism": "Prunus persica",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000006882%29",
        "priority": "bassa",
        "category": "scarti_bucce_frutta",
        "est_proteins": 38732,
        "proteome_id": "UP000006882",
        "note": "NUOVO. [REF] Noccioli e bucce."
    },
    "Prunus_avium": {
        "name": "Ciliegia (nocciolo)",
        "organism": "Prunus avium",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000515124%29",
        "priority": "bassa",
        "category": "scarti_bucce_frutta",
        "est_proteins": 30515,
        "proteome_id": "UP000515124",
        "note": "NUOVO. Noccioli di ciliegia."
    },
    "Brassica_oleracea": {
        "name": "Broccolo (scarti)",
        "organism": "Brassica oleracea var. oleracea",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000032141%29",
        "priority": "alta",
        "category": "scarti_bucce_ortaggi",
        "est_proteins": 58535,
        "proteome_id": "UP000032141",
        "note": "FIX: var. italica aveva 243. Usa var. oleracea (stessa specie). Sulforafano."
    },
    "Daucus_carota": {
        "name": "Carota (scarti)",
        "organism": "Daucus carota",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000077755%29",
        "priority": "media",
        "category": "scarti_bucce_ortaggi",
        "est_proteins": 35531,
        "proteome_id": "UP000077755",
        "note": "NUOVO. [REF] Scarti di carota."
    },
    "Cucurbita_pepo": {
        "name": "Zucchina (scarti)",
        "organism": "Cucurbita pepo",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000504609%29",
        "priority": "bassa",
        "category": "scarti_bucce_ortaggi",
        "est_proteins": 35465,
        "proteome_id": "UP000504609",
        "note": "NUOVO. Bucce e semi."
    },
    "Beta_vulgaris": {
        "name": "Barbabietola (scarti)",
        "organism": "Beta vulgaris",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000035740%29",
        "priority": "bassa",
        "category": "scarti_bucce_ortaggi",
        "est_proteins": 7679,
        "proteome_id": "UP000035740",
        "note": "NUOVO. Betalaine note."
    },
    "Capsicum_annuum": {
        "name": "Peperone (scarti)",
        "organism": "Capsicum annuum",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000222542%29",
        "priority": "bassa",
        "category": "scarti_bucce_ortaggi",
        "est_proteins": 35548,
        "proteome_id": "UP000222542",
        "note": "NUOVO. [REF] Capsaicina nota."
    },
    "Juglans_regia": {
        "name": "Noce (guscio/mallo)",
        "organism": "Juglans regia",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000619265%29",
        "priority": "media",
        "category": "scarti_gusci_semi",
        "est_proteins": 39594,
        "proteome_id": "UP000619265",
        "note": "NUOVO. Gusci e mallo di noce."
    },
    "Prunus_dulcis": {
        "name": "Mandorla (guscio)",
        "organism": "Prunus dulcis",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP001054821%29",
        "priority": "media",
        "category": "scarti_gusci_semi",
        "est_proteins": 44957,
        "proteome_id": "UP001054821",
        "note": "NUOVO. [REF] Gusci di mandorla."
    },
    "Coffea_arabica": {
        "name": "Caffè (fondi)",
        "organism": "Coffea arabica",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP001652660%29",
        "priority": "alta",
        "category": "scarti_gusci_semi",
        "est_proteins": 65937,
        "proteome_id": "UP001652660",
        "note": "NUOVO. Fondi di caffè, enorme volume di scarto."
    },
    "Theobroma_cacao": {
        "name": "Cacao (guscio)",
        "organism": "Theobroma cacao",
        "url": "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=%28proteome%3AUP000026915%29",
        "priority": "media",
        "category": "scarti_gusci_semi",
        "est_proteins": 40609,
        "proteome_id": "UP000026915",
        "note": "NUOVO. [REF] Gusci di cacao."
    }
}

# Applica selezione proteomi (basata su DATABASE scelto in cell 2)
active_proteomes = {}
for k, v in PROTEOMES.items():
    cat = v.get("category", "")
    if cat in SELECTED_CATEGORIES:
        active_proteomes[k] = v

if not active_proteomes:
    print("⚠️ Nessun proteoma matchato — carico tutti")
    active_proteomes = PROTEOMES

n_active = len(active_proteomes)
n_total = len(PROTEOMES)
total_prot = sum(v.get("est_proteins", 0) for v in active_proteomes.values())
print(f"\n🌱 Proteomi attivi: {n_active} su {n_total}")
for k, v in active_proteomes.items():
    print(f"   {v['name']:25s} {v['category']:25s} ~{v.get('est_proteins',0):>8,}")
print(f"\n📦 Totale stimato: {total_prot:,} proteine")



🌱 Proteomi attivi: 22 su 61
   Pleurotus                 fito_funghi               ~  11,624
   Cordyceps                 fito_funghi               ~   9,651
   Cordyceps sinensis        fito_funghi               ~   9,923
   Maitake                   fito_funghi               ~  14,985
   Shiitake                  fito_funghi               ~  12,046
   Coda di tacchino          fito_funghi               ~   1,022
   Reishi                    fito_funghi               ~     360
   Zenzero                   fito_piante               ~  71,836
   Ginseng                   fito_piante               ~     840
   Tè verde                  fito_piante               ~  30,052
   Artemisia                 fito_piante               ~  66,067
   Olivo                     fito_piante               ~  78,352
   Curcuma                   fito_piante               ~     244
   Liquirizia                fito_piante               ~     346
   Moringa                   fito_piante               ~     

## 📂 STEP 2 — Load Proteomes
Load proteomes from the downloaded databases (tar.gz). If missing, you can upload them locally.

In [22]:
# ============================================================
# STEP 2 — CARICA PROTEOMI (SOLO LOCALE E FIX ESTRAZIONE)
# ============================================================
import os, glob, subprocess

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

def parse_fasta(filepath):
    """Parser FASTA. Yield (header, sequence)."""
    header, seq_parts = "", []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if header:
                    yield header, "".join(seq_parts)
                header = line[1:]
                seq_parts = []
            else:
                seq_parts.append(line.upper())
    if header:
        yield header, "".join(seq_parts)

# Cerca tar.gz SOLO in locale (/content o WORK_DIR)
print("🔍 Searching for tar.gz databases locally (ignoring Drive)...\n")
tar_files = []
for search in ["/content", WORK_DIR]:
    if search and os.path.exists(search):
        tar_files.extend(glob.glob(os.path.join(search, "db_*.tar.gz")))

tar_files = list(set(tar_files))  # deduplica

if tar_files:
    for tf in tar_files:
        print(f"   📦 Extracting {os.path.basename(tf)}...")
        # Estrae nella cartella dati
        os.system(f'tar xf "{tf}" -C {DATA_DIR} 2>/dev/null')
        # FORZATURA: sposta tutti i .fasta estratti in sottocartelle direttamente in DATA_DIR
        os.system(f'find {DATA_DIR} -mindepth 2 -type f -name "*.fasta" -exec mv {{}} {DATA_DIR}/ \; 2>/dev/null')
        print(f"   ✅ Completed")
else:
    print("❌ No db_*.tar.gz found in /content!")
    if ENV == 'colab':
        from google.colab import files
        print("📂 Upload the db_fitoterapici.tar.gz file:")
        uploaded = files.upload()
        for fname in uploaded.keys():
            os.system(f'tar xf "{fname}" -C {DATA_DIR} 2>/dev/null')
            os.system(f'find {DATA_DIR} -mindepth 2 -type f -name "*.fasta" -exec mv {{}} {DATA_DIR}/ \; 2>/dev/null')
            print(f"   ✅ {fname}")
    else:
        raise FileNotFoundError("No database found.")

# Conta fasta e verifica presenza
proteome_stats = {}
for fpath in sorted(glob.glob(os.path.join(DATA_DIR, "*.fasta"))):
    pname = os.path.basename(fpath).replace(".fasta", "")
    if pname in active_proteomes:
        n_seq = sum(1 for line in open(fpath) if line.startswith(">"))
        if n_seq > 0:
            proteome_stats[pname] = n_seq
            name = active_proteomes[pname].get("name", pname)
            print(f"   ✅ {name:25s} {n_seq:>8,} proteins")
        else:
            print(f"   ⚠️ {pname}: empty file!")

missing = [k for k in active_proteomes if k not in proteome_stats]
if missing:
    print(f"\n⚠️ Missing {len(missing)} fasta (maybe not in tar.gz):")
    for m in missing:
        print(f"   ❌ {active_proteomes[m]['name']} ({m})")

total_proteins = sum(proteome_stats.values())
print(f"\n📦 Actual total loaded: {total_proteins:,} proteins from {len(proteome_stats)} proteomes")
if len(proteome_stats) > 0:
    print("✅ Ready for STEP 3!")
else:
    raise FileNotFoundError("No fasta loaded! Check that the tar.gz file contains the correct .fasta files.")


🔍 Searching for tar.gz databases locally (ignoring Drive)...

   📦 Extracting db_fitoterapici.tar.gz...


<>:39: SyntaxWarning: invalid escape sequence '\;'
<>:49: SyntaxWarning: invalid escape sequence '\;'
<>:39: SyntaxWarning: invalid escape sequence '\;'
<>:49: SyntaxWarning: invalid escape sequence '\;'
/tmp/ipykernel_3631/963682051.py:39: SyntaxWarning: invalid escape sequence '\;'
  os.system(f'find {DATA_DIR} -mindepth 2 -type f -name "*.fasta" -exec mv {{}} {DATA_DIR}/ \; 2>/dev/null')
/tmp/ipykernel_3631/963682051.py:49: SyntaxWarning: invalid escape sequence '\;'
  os.system(f'find {DATA_DIR} -mindepth 2 -type f -name "*.fasta" -exec mv {{}} {DATA_DIR}/ \; 2>/dev/null')


   ✅ Completed
   ✅ Artemisia                   66,067 proteins
   ✅ Astragalo                       76 proteins
   ✅ Tè verde                    30,052 proteins
   ✅ Centella                       170 proteins
   ✅ Cannella                       172 proteins
   ✅ Cordyceps                    9,651 proteins
   ✅ Zafferano                      315 proteins
   ✅ Curcuma                        244 proteins
   ✅ Reishi                         360 proteins
   ✅ Liquirizia                     346 proteins
   ✅ Maitake                     14,985 proteins
   ✅ Shiitake                    12,046 proteins
   ✅ Moringa                        192 proteins
   ✅ Olivo                       78,352 proteins
   ✅ Cordyceps sinensis           9,923 proteins
   ✅ Ginseng                        840 proteins
   ✅ Pleurotus                   11,624 proteins
   ✅ Poligono del Giappone          185 proteins
   ✅ Cardo mariano                  183 proteins
   ✅ Coda di tacchino             1,022 proteins
   ✅ 

## 🧪 STEP 3 — In Silico Enzymatic Hydrolysis
Instead of a brute-force sliding window, we simulate cleavage with **6 real enzymes**:
- **Trypsin**: cleaves after K, R
- **Pepsin**: cleaves after F, Y, W, L
- **Chymotrypsin**: cleaves after F, Y, W
- **Papain**: cleaves after R, K, Q, H, G, Y, F, W (broad)
- **Bromelain**: cleaves after K, R, A, Y
- **Alcalase**: cleaves after F, W, Y, L, I, V, M, A (broad)

This reduces fragments by **~97%** and generates only industrially producible peptides.

In [ ]:
# ============================================================
# STEP 3 — IN SILICO ENZYMATIC HYDROLYSIS (6 ENZYMES)
# ============================================================
import csv
from tqdm.auto import tqdm
from collections import defaultdict

# Enzyme specificity: enzyme cleaves AFTER these residues
ENZYMES = {
    "Trypsin":      set("KR"),
    "Pepsin":       set("FYWL"),
    "Chymotrypsin": set("FYW"),
    "Papain":       set("RKQHGYFW"),
    "Bromelain":    set("KRAY"),
    "Alcalase":     set("FWYLIVMA"),
}

# Parameters
MIN_LEN = 6
MAX_LEN = 20
AROMATIC = set("FWYH")
BASIC = set("RKH")

def enzymatic_digest(protein_seq, enzyme_cuts):
    """
    Simulates enzymatic cleavage of a protein.
    The enzyme cleaves AFTER the specified residues.
    Returns a list of (peptide, start_pos).
    """
    fragments = []
    start = 0
    for i, aa in enumerate(protein_seq):
        if aa in enzyme_cuts:
            frag = protein_seq[start:i+1]
            if MIN_LEN <= len(frag) <= MAX_LEN:
                fragments.append((frag, start))
            start = i + 1
    # Last fragment
    frag = protein_seq[start:]
    if MIN_LEN <= len(frag) <= MAX_LEN:
        fragments.append((frag, start))
    return fragments

def passes_biofilter(peptide):
    """Biochemical filter: at least 1 aromatic AND 1 basic, only standard AAs."""
    if not all(aa in VALID_AA for aa in peptide):
        return False
    has_aromatic = any(aa in AROMATIC for aa in peptide)
    has_basic = any(aa in BASIC for aa in peptide)
    return has_aromatic and has_basic

# ── Processing ──
PEPTIDES_CSV = os.path.join(WORK_DIR, "peptides_hydrolysis.csv")
seen_peptides = set()
total_peptides = 0
stats_per_proteome = defaultdict(int)
stats_per_enzyme = defaultdict(int)

print("🧪 In silico enzymatic hydrolysis...\n")

with open(PEPTIDES_CSV, "w", newline="") as f_out:
    writer = csv.writer(f_out)
    writer.writerow(["peptide", "length", "protein_id", "protein_name",
                     "source", "position", "enzyme"])

    for pname, pdata in active_proteomes.items():
        fpath = os.path.join(DATA_DIR, f"{pname}.fasta")
        if not os.path.exists(fpath):
            continue

        n_seq = proteome_stats.get(pname, 0)
        pep_count = 0

        for header, seq in tqdm(parse_fasta(fpath), total=n_seq,
                                desc=f"🧪 {pdata['name'][:20]:20s}"):
            clean = "".join(c for c in seq if c in VALID_AA)
            if len(clean) < MIN_LEN:
                continue

            prot_id = header.split("|")[1] if "|" in header else header.split()[0]
            prot_name = header.split("|")[2].split(" OS=")[0] if "|" in header and "OS=" in header else header[:80]

            # Digestion with ALL enzymes
            for enz_name, enz_cuts in ENZYMES.items():
                for frag, start_pos in enzymatic_digest(clean, enz_cuts):
                    if frag in seen_peptides:
                        continue
                    if not passes_biofilter(frag):
                        continue
                    seen_peptides.add(frag)
                    writer.writerow([frag, len(frag), prot_id, prot_name,
                                    pname, f"{start_pos+1}-{start_pos+len(frag)}",
                                    enz_name])
                    total_peptides += 1
                    pep_count += 1
                    stats_per_enzyme[enz_name] += 1

        stats_per_proteome[pname] = pep_count

# Report
print(f"\n{'='*60}")
print(f"✅ Hydrolysis completed! {total_peptides:,} unique peptides generated")
print(f"\n📊 By proteome:")
for pname, count in sorted(stats_per_proteome.items(), key=lambda x: -x[1]):
    print(f"   {active_proteomes[pname]['name']:25s} → {count:>8,} peptides")
print(f"\n🧪 By enzyme:")
for enz, count in sorted(stats_per_enzyme.items(), key=lambda x: -x[1]):
    print(f"   {enz:15s} → {count:>8,} peptides")
print(f"\n💾 Saved to: {PEPTIDES_CSV}")


🧪 In silico enzymatic hydrolysis...



🧪 Pleurotus           :   0%|          | 0/11624 [00:00<?, ?it/s]

🧪 Cordyceps           :   0%|          | 0/9651 [00:00<?, ?it/s]

🧪 Cordyceps sinensis  :   0%|          | 0/9923 [00:00<?, ?it/s]

🧪 Maitake             :   0%|          | 0/14985 [00:00<?, ?it/s]

...

🧪 Zenzero             :   0%|          | 0/71836 [00:00<?, ?it/s]

🧪 Ginseng             :   0%|          | 0/840 [00:00<?, ?it/s]

🧪 Tè verde            :   0%|          | 0/30052 [00:00<?, ?it/s]

🧪 Artemisia           :   0%|          | 0/66067 [00:00<?, ?it/s]

🧪 Olivo               :   0%|          | 0/78352 [00:00<?, ?it/s]

## STEP 3b — Single-Filter Mode (Table S2 for SPAFE paper)

**Difference vs standard pipeline:**
- Standard (early-exit): peptide is dropped at the first failed filter → not tested on the others.
- Single-filter mode: each filter is applied independently to the total → 6 standalone numbers.

Output: `filter_counts.json` + `filter_counts.txt` to forward to Minoo for Table S2.

No ESM-2, no GPU, no Boltz. Only statistical counts on the `peptides_hydrolysis.csv` just generated from STEP 3.

In [24]:
# ============================================================
# STEP 3b — SINGLE-FILTER & CASCADING MODE (Table S2)
# Calculates both independent passes and cascading reduction.
# ============================================================
import csv, os, json
from tqdm.auto import tqdm

# ── Physicochemical constants (identical to STEP 3b pipeline v9) ──
# Radzicka & Wolfenden 1988: ΔG transfer cyclohexane → water (kcal/mol)
BOMAN_TRANSFER = {
    "A": 0.50, "R": -2.53, "N": -6.64, "D": -8.72, "C": -1.29,
    "Q": -5.54, "E": -6.81, "G": 1.15, "H": -4.66, "I": 4.92,
    "K": -5.55, "L": 4.92, "M": 2.35, "F": 2.98, "P": -0.02,
    "S": -3.40, "T": -2.57, "W": 2.33, "Y": -0.14, "V": 4.04,
}
CHARGE_AA = {"R": 1, "K": 1, "H": 0.5, "D": -1, "E": -1}
HYDRO_KD = {"A":1.8,"R":-4.5,"N":-3.5,"D":-3.5,"C":2.5,"Q":-3.5,"E":-3.5,"G":-0.4,
            "H":-3.2,"I":4.5,"L":3.8,"K":-3.9,"M":1.9,"F":2.8,"P":-1.6,"S":-0.8,
            "T":-0.7,"W":-0.9,"Y":-1.3,"V":4.2}
MW_AA = {"A":89.1,"R":174.2,"N":132.1,"D":133.1,"C":121.2,"Q":146.2,"E":147.1,"G":75.0,
         "H":155.2,"I":131.2,"L":131.2,"K":146.2,"M":149.2,"F":165.2,"P":115.1,"S":105.1,
         "T":119.1,"W":204.2,"Y":181.2,"V":117.1}

# Instability Index DIWV (Guruprasad 1990)
DIWV = {"WW":1,"WC":1,"WM":24.68,"WH":24.68,"WY":1,"WF":1,"WQ":1,"WN":13.34,
        "WE":1,"WD":-14.03,"WK":1,"WR":-14.03,"WS":1,"WT":-14.03,"WG":-9.37,
        "WA":-14.03,"WV":-7.49,"WL":13.34,"WI":1,"WP":-1.88,
        "CW":24.68,"CC":1,"CM":33.6,"CH":33.6,"CY":1,"CF":-14.03,"CQ":-6.54,
        "CN":1,"CE":1,"CD":20.26,"CK":1,"CR":1,"CS":1,"CT":33.6,"CG":1,
        "CA":1,"CV":-6.54,"CL":20.26,"CI":1,"CP":20.26,
        "AW":1,"AC":44.94,"AM":1,"AH":-7.49,"AY":1,"AF":1,"AQ":1,"AN":1,
        "AE":1,"AD":-7.49,"AK":1,"AR":1,"AS":1,"AT":1,"AG":1,
        "AA":1,"AV":1,"AL":1,"AI":1,"AP":20.26,
        "GW":13.34,"GC":1,"GM":1,"GH":1,"GY":-7.49,"GF":1,"GQ":1,"GN":-7.49,
        "GE":-6.54,"GD":1,"GK":-7.49,"GR":1,"GS":1,"GT":-7.49,"GG":1,
        "GA":1,"GV":1,"GL":1,"GI":-7.49,"GP":1}

def boman_index(seq):
    total = sum(BOMAN_TRANSFER.get(aa, 0) for aa in seq)
    return -total / len(seq) if seq else 0

def net_charge(seq):
    return sum(CHARGE_AA.get(aa, 0) for aa in seq)

def gravy(seq):
    vals = [HYDRO_KD.get(aa, 0) for aa in seq]
    return sum(vals) / len(vals) if vals else 0

def mol_weight(seq):
    return sum(MW_AA.get(aa, 0) for aa in seq) - (len(seq) - 1) * 18.02

def instability_index(seq):
    n = len(seq)
    if n < 2:
        return 100
    total = 0
    for j in range(n - 1):
        dipep = seq[j] + seq[j+1]
        total += DIWV.get(dipep, 1)
    return (10.0 / n) * total

def hydrophobic_ratio(seq):
    HYDROPHOBIC = set("AILMFPWV")
    return sum(1 for aa in seq if aa in HYDROPHOBIC) / len(seq) if seq else 0

# ── Final Thresholds (identical to STEP 3b pipeline v9) ──
MIN_BOMAN = 1.0
MIN_GRAVY, MAX_GRAVY = -2.0, 0.5
MIN_CHARGE, MAX_CHARGE = -2, 4
MAX_INSTAB = 40
MAX_MW = 1500
MIN_HYDRO, MAX_HYDRO = 0.30, 0.60

# ── Read CSV ──
print(f"📂 Reading {PEPTIDES_CSV}...")
with open(PEPTIDES_CSV, "r") as f:
    all_peptides = [row["peptide"] for row in csv.DictReader(f)]

total = len(all_peptides)
print(f"   Total peptides: {total:,}\n")

# ── Single-filter counting (Independent) ──
counts_independent = {"boman": 0, "gravy": 0, "charge": 0, "instab": 0, "mw": 0, "hydro": 0}

print("🔬 Calculating standalone counts (independent)...")
for seq in tqdm(all_peptides):
    if boman_index(seq) >= MIN_BOMAN: counts_independent["boman"] += 1
    if MIN_GRAVY <= gravy(seq) <= MAX_GRAVY: counts_independent["gravy"] += 1
    if MIN_CHARGE <= net_charge(seq) <= MAX_CHARGE: counts_independent["charge"] += 1
    if instability_index(seq) < MAX_INSTAB: counts_independent["instab"] += 1
    if mol_weight(seq) <= MAX_MW: counts_independent["mw"] += 1
    if MIN_HYDRO <= hydrophobic_ratio(seq) <= MAX_HYDRO: counts_independent["hydro"] += 1

# ── Cascading filtering (Sequential) ──
print("\n🌊 Calculating cascading counts (sequential)...")
cascading_pool = all_peptides
cascading_counts = {}
cascading_dropped = {}

# 1. Boman
pass_boman = [seq for seq in cascading_pool if boman_index(seq) >= MIN_BOMAN]
cascading_counts['boman'] = len(pass_boman)
cascading_dropped['boman'] = len(cascading_pool) - len(pass_boman)
cascading_pool = pass_boman

# 2. GRAVY
pass_gravy = [seq for seq in cascading_pool if MIN_GRAVY <= gravy(seq) <= MAX_GRAVY]
cascading_counts['gravy'] = len(pass_gravy)
cascading_dropped['gravy'] = len(cascading_pool) - len(pass_gravy)
cascading_pool = pass_gravy

# 3. Charge
pass_charge = [seq for seq in cascading_pool if MIN_CHARGE <= net_charge(seq) <= MAX_CHARGE]
cascading_counts['charge'] = len(pass_charge)
cascading_dropped['charge'] = len(cascading_pool) - len(pass_charge)
cascading_pool = pass_charge

# 4. Instab
pass_instab = [seq for seq in cascading_pool if instability_index(seq) < MAX_INSTAB]
cascading_counts['instab'] = len(pass_instab)
cascading_dropped['instab'] = len(cascading_pool) - len(pass_instab)
cascading_pool = pass_instab

# 5. MW
pass_mw = [seq for seq in cascading_pool if mol_weight(seq) <= MAX_MW]
cascading_counts['mw'] = len(pass_mw)
cascading_dropped['mw'] = len(cascading_pool) - len(pass_mw)
cascading_pool = pass_mw

# 6. Hydro
pass_hydro = [seq for seq in cascading_pool if MIN_HYDRO <= hydrophobic_ratio(seq) <= MAX_HYDRO]
cascading_counts['hydro'] = len(pass_hydro)
cascading_dropped['hydro'] = len(cascading_pool) - len(pass_hydro)
cascading_pool = pass_hydro

passed_all = len(cascading_pool)

# ── Report ──
print(f"\n{'='*75}")
print(f"  CASCADING FILTERING (SEQUENTIAL)")
print(f"{'='*75}")
print(f"  {'Filter':<28} {'Remaining':>12} {'Dropped':>12} {'Remaining %':>12}")
print(f"  {'-'*75}")
print(f"  {'Initial Input':<28} {total:>12,} {'-':>12} {100.0:>11.2f}%")

labels = {
    "boman":  f"1. Boman >= {MIN_BOMAN}",
    "gravy":  f"2. GRAVY [{MIN_GRAVY},{MAX_GRAVY}]",
    "charge": f"3. Charge [{MIN_CHARGE},{MAX_CHARGE}]",
    "instab": f"4. Instab. < {MAX_INSTAB}",
    "mw":     f"5. MW <= {MAX_MW}",
    "hydro":  f"6. Hydrophob. [{MIN_HYDRO},{MAX_HYDRO}]",
}

for key, lbl in labels.items():
    n_rem = cascading_counts[key]
    n_drop = cascading_dropped[key]
    pct = 100 * n_rem / total
    print(f"  {lbl:<28} {n_rem:>12,} {n_drop:>12,} {pct:>11.2f}%")
print(f"  {'-'*75}")
print(f"  {'FINAL Combined':<28} {passed_all:>12,} {'-':>12} {100*passed_all/total:>11.2f}%")

print(f"\n{'='*75}")
print(f"  INDEPENDENT FILTERING (STANDALONE)")
print(f"{'='*75}")
print(f"  {'Filter':<28} {'Passed (n)':>12} {'Passed %':>12}")
print(f"  {'-'*75}")
for key, lbl in labels.items():
    n_ind = counts_independent[key]
    pct_ind = 100 * n_ind / total
    print(f"  {lbl[3:]:<28} {n_ind:>12,} {pct_ind:>11.2f}%")
print(f"{'='*75}")

# ── Save output for Publication / Minoo ──
OUT_JSON = os.path.join(WORK_DIR, "filter_counts.json")
OUT_TXT = os.path.join(WORK_DIR, "filter_counts.txt")

result = {
    "total_input": total,
    "cascading_pass_counts": cascading_counts,
    "cascading_dropped_counts": cascading_dropped,
    "standalone_pass_counts": counts_independent,
    "combined_pass_count": passed_all,
    "thresholds": {
        "boman_min": MIN_BOMAN,
        "gravy_min": MIN_GRAVY, "gravy_max": MAX_GRAVY,
        "charge_min": MIN_CHARGE, "charge_max": MAX_CHARGE,
        "instab_max": MAX_INSTAB,
        "mw_max": MAX_MW,
        "hydro_min": MIN_HYDRO, "hydro_max": MAX_HYDRO,
    },
}
with open(OUT_JSON, "w") as f:
    json.dump(result, f, indent=2)

with open(OUT_TXT, "w") as f:
    f.write(f"Peptide Filtering — NutrAI Pipeline Table S2\n")
    f.write(f"Total input: {total:,}\n\n")
    f.write(f"--- CASCADING (SEQUENTIAL) ---\n")
    for key, lbl in labels.items():
        f.write(f"{lbl:<28} Remaining: {cascading_counts[key]:>10,} | Dropped: {cascading_dropped[key]:>10,}\n")
    f.write(f"\n--- INDEPENDENT (STANDALONE) ---\n")
    for key, lbl in labels.items():
        f.write(f"{lbl[3:]:<28} Passed: {counts_independent[key]:>10,}\n")

print(f"\n💾 Output saved:")
print(f"   {OUT_JSON}")
print(f"   {OUT_TXT}")


📂 Reading /workspace/nutrai_pipeline/peptides_hydrolysis.csv...
   Total peptides: 8,645,509

🔬 Calculating standalone counts (independent)...


  0%|          | 0/8645509 [00:00<?, ?it/s]


🌊 Calculating cascading counts (sequential)...

  CASCADING FILTERING (SEQUENTIAL)
  Filter                          Remaining      Dropped  Remaining %
  ---------------------------------------------------------------------------
  Initial Input                   8,645,509            -      100.00%
  1. Boman >= 1.0                 5,128,049    3,517,460       59.31%
  2. GRAVY [-2.0,0.5]             4,573,283      554,766       52.90%
  3. Charge [-2,4]                4,091,058      482,225       47.32%
  4. Instab. < 40                 3,988,394      102,664       46.13%
  5. MW <= 1500                   2,864,569    1,123,825       33.13%
  6. Hydrophob. [0.3,0.6]         1,643,193    1,221,376       19.01%
  ---------------------------------------------------------------------------
  FINAL Combined                  1,643,193            -       19.01%

  INDEPENDENT FILTERING (STANDALONE)
  Filter                         Passed (n)     Passed %
  ---------------------------------